# Checkpoint A solutions -- the Part-1 recommender

These reference solutions close Part 1. Across seven questions we assemble the four Part-1 signal
paths (plus the random floor from Unit 1) into one calibrated blend, measure it the way a real system
is measured (precision@k, NDCG@k, coverage, diversity, novelty), read a leave-one-path-out ablation,
and finally give the frozen recommender a single honest grade on the sealed `test` split.

The discipline of the checkpoint is **freeze, then score**. Questions 1-6 run entirely on the
`val` split: we search the catalog, fit the four signal paths, read the deepened metrics, build
the blend at the pinned config, and ablate it. Question 5 FREEZES every decision into
`FROZEN_PATHS`, `FROZEN_WEIGHTS`, and `FROZEN_POOL`. Question 7 is the **only** cell that touches
`split="test"`: it scores exactly the frozen recommender once and interprets val-versus-test -- it
never returns to selection, because a holdout you tune against stops being a holdout.

Everything is seeded (`SEED = 0`) and measured at `K = 10` with the 60 cold readers excluded, so
the notebook runs top-to-bottom deterministically; each question ends with an `assert` on a
computed result so a wrong number fails loudly rather than printing quietly. The honest thesis,
stated up front and never overclaimed below: on this data at this pinned config the blend beats the
best single path (MF, ~0.276) on hit@10 (~0.306) AND covers about 4.5x more of the catalog (~0.333
vs ~0.0745) -- but that is a result here, not a law that "a blend always wins", and the ablation
shows "every path contributes" is false (dropping popularity does not lower hit@10).

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    PathRegistry,
    blend,
    catalog_coverage,
    hit_rate_at_k,
    intra_list_diversity,
    ndcg_at_k,
    novelty,
    precision_at_k,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
    search_catalog,
)

SEED = 0
K = 10   # every metric is reported at the top-10 cutoff

# The same generated substrate Units 1-5 ran on.
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)        # the 2000-book catalog: the coverage denominator
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])   # 60 readers no personalized path can serve

# The interaction log as plain row dicts (the row-mapping log every path duck-types).
rows = pd.read_csv(interactions_path).to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set; their `val` positives are
# the held-out relevant set. The `test` split stays SEALED until Question 7.
seen_by = defaultdict(set)
val_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_by[row["reader_id"]].add(row["item_id"])

warm_readers = sorted(reader for reader in val_by if reader not in cold_readers)
print(f"catalog books: {len(catalog_ids)}   cold readers: {len(cold_readers)}   "
      f"warm val readers: {len(warm_readers)}")

## Question 1 -- Know the shelf: search the catalog

Every recommender draws from a fixed **catalog** -- the universe of items it can return -- and that
same catalog is the denominator when we later ask how much of the shelf a recommender reaches. Use
the Unit-1 `search_catalog` helper to explore it.

Find every book in the `fantasy` genre, and (separately) every book published in `2011`. Report how
many of each there are, and the title of one catalog book so you can see the schema. Then, in one or
two sentences, state what the catalog is in this system and why its size is the right denominator for
a coverage metric.

In [ ]:
# The catalog is a {item_id: Book} mapping; search_catalog filters it on the real schema.
fantasy_books = search_catalog(catalog, genre="fantasy")   # list of Book objects
n_fantasy = len(fantasy_books)

books_2011 = search_catalog(catalog, year=2011)
n_2011 = len(books_2011)

sample_title = catalog[catalog_ids[0]].title   # one book's title -- see the schema
catalog_size = len(catalog_ids)

catalog_role = (
    "The catalog is the fixed universe of items every retrieval path draws from -- the shelf the "
    "recommender is allowed to return. Its size is the right denominator for catalog coverage "
    "because coverage asks what fraction of THAT shelf the system ever surfaces, so the whole "
    "catalog, not just the books that happen to get recommended, sets the 100% mark."
)

print(f"fantasy books: {n_fantasy}")
print(f"books from 2011: {n_2011}")
print(f"catalog size: {catalog_size}   sample title: {sample_title!r}")

# The catalog has a known shape on this seeded substrate.
assert catalog_size == 2000, "the Part-1 catalog is the 2000-book shelf"
assert n_fantasy == 350, "fantasy is a real, non-empty genre slice"
assert n_2011 == 25, "2011 is a real, non-empty publication year"
assert 0 < n_fantasy < catalog_size and 0 < n_2011 < catalog_size

## Question 2 -- Assemble the Part-1 paths

A recommender is built from retrieval paths, each a `RetrievalPath` with the same `fit` / `retrieve`
contract. Build and `fit` the four Part-1 signal paths on the interaction log -- **popularity** (Unit
2), **lexical** (Unit 3), **item-item CF** (Unit 4), and **matrix factorization** (Unit 5; the MF fit
is the only slow one, about 15 s) -- and register all four in a `PathRegistry` so later questions can
loop over them by name. Then, as a sanity check, produce one warm reader's top-10 from a single path
with the retrieve-then-rank pattern, dropping the books that reader has already seen.

In [ ]:
# Four fits in this cell (the MF fit is the only slow one, ~15 s); later questions REUSE them.
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = bookrec.MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)

registry = PathRegistry()
for path in (popularity, lexical, item_item, mf):
    registry.register(path)
registered_names = [path.name for path in registry]   # insertion order

# Sanity check: one warm reader's top-10 from the item-item path (retrieve-then-rank).
reader_id = warm_readers[0]
seen = seen_by[reader_id]
sample_recs = rank(item_item.retrieve(reader_id, {"seen": seen}, K), n=K, exclude=seen)
sample_rec_ids = [candidate.item_id for candidate in sample_recs]

print(f"registered paths: {registered_names}")
print(f"warm reader {reader_id} item-item top-{K}: {sample_rec_ids}")

# All four Part-1 signal paths are registered under their stable blend names, in order.
assert registered_names == ["popularity", "lexical", "item-item", "mf"]
assert len(sample_rec_ids) == K, "retrieve-then-rank returns a full top-k shortlist"
# retrieve-then-rank must never recommend a book the reader has already seen.
assert not (set(sample_rec_ids) & seen), "the top-k must exclude the reader's seen books"

## Question 3 -- Ranking metrics: precision@k and NDCG@k (not RMSE)

Hit@k only asks whether *any* relevant book reached the top k. Two richer **ranking metrics** look
deeper: **precision@k** is the fraction of the top-k slots that are relevant, and **NDCG@k**
additionally rewards putting a relevant book *higher* in the list. The frozen scoreboard reports all
of these; score each registered path on `val` with `run_validation_scoreboard` and collect its
hit@10, precision@10, and NDCG@10.

Then answer, in two or three sentences: this is an implicit-feedback top-k problem (a reader either
engaged a book or did not, and only ever sees a ranked shortlist). Why is a ranking metric the right
lens here rather than rating-prediction RMSE, and what does NDCG@k reward that hit@k and precision@k
both ignore?

In [ ]:
ranking_rows = []
for path in registry:                  # popularity, lexical, item-item, mf
    board = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers,
    )                                   # split defaults to "val" -- do NOT pass split here
    ranking_rows.append({
        "path": path.name,
        "hit@10": board.hit_rate_at_k,
        "precision@10": board.precision_at_k,
        "ndcg@10": board.ndcg_at_k,
    })

by_hit = {row["path"]: row["hit@10"] for row in ranking_rows}
best_single = max(by_hit, key=by_hit.get)

ranking_metrics_explanation = (
    "This is implicit feedback: a reader either engaged a book or did not, and there are no "
    "held-out star RATINGS to take a squared error against, so RMSE has nothing to score -- a "
    "ranking metric is the right lens. The reader only ever scans a ranked shortlist, so the "
    "question is always 'is a relevant book near the top?'. Precision@k counts how many of the k "
    "slots are relevant, but like hit@k it is order-blind within the top k; NDCG@k is the only one "
    "of the three that rewards placing a relevant book HIGHER, discounting its gain by rank."
)

print(f"{'path':12s} {'hit@10':>8s} {'prec@10':>8s} {'ndcg@10':>8s}")
for row in ranking_rows:
    print(f"{row['path']:12s} {row['hit@10']:8.3f} {row['precision@10']:8.3f} {row['ndcg@10']:8.3f}")
print(f"best single path on hit@10: {best_single} ({by_hit[best_single]:.3f})")

# MF is the strongest single path on hit@10 (~0.276), ahead of item-item (~0.252).
assert best_single == "mf", "MF should top the single-path hit@10 board"
assert abs(by_hit["mf"] - 0.276) < 0.02, "MF single-path hit@10 is about 0.276"
# Every path reports a finite precision and NDCG in [0, 1] -- the deepened metrics are live.
for row in ranking_rows:
    assert 0.0 <= row["precision@10"] <= 1.0 and 0.0 <= row["ndcg@10"] <= 1.0

## Question 4 -- Beyond accuracy: coverage, diversity, novelty

Accuracy is not the whole story. **Catalog coverage** is the fraction of the catalog a path ever
recommends to anyone; **intra-list diversity** of a reader's top-k is `1 - mean pairwise similarity`;
**novelty** is the mean self-information `-log2 p_hat` of the recommended books (rarer books are more
novel). Reusing the paths fitted in Question 2, measure the catalog coverage of matrix factorization
and of item-item CF, and for one warm reader compute the intra-list diversity of the item-item top-10
(using the fitted item-item cosine as the book-book similarity) and the novelty of both the item-item
and the popularity top-10.

Then, in two or three sentences, interpret what you find: MF ranks best (Question 3) yet reaches only
a sliver of the catalog, item-item reaches far more, and popularity's picks are the least novel. Why
can two paths with similar hit@k still serve very different readers?

In [ ]:
def coverage_of(name, path):
    # A single-path blend is just that path; the blended scoreboard reports its catalog coverage.
    return run_blended_scoreboard(
        {name: path}, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=30, weights={name: 1.0}, cold_readers=cold_readers,
    ).catalog_coverage


mf_coverage = coverage_of("mf", mf)                       # narrow (~0.0745)
item_item_coverage = coverage_of("item-item", item_item)  # broad (~0.49)

# The fitted item-item cosine, addressed by item id, is the book-book similarity for diversity.
sim_col = {item_id: j for j, item_id in enumerate(item_item.artifact()["item_ids"])}
similarity = item_item.similarity   # the 2000 x 2000 cosine matrix

reader = warm_readers[0]
reader_seen = seen_by[reader]
item_item_ids = [
    c.item_id
    for c in rank(item_item.retrieve(reader, {"seen": reader_seen}, 30), n=K, exclude=reader_seen)
]
popularity_ids = [
    c.item_id
    for c in rank(popularity.retrieve(reader, {"seen": reader_seen}, 30), n=K, exclude=reader_seen)
]

diversity_item_item = intra_list_diversity(
    item_item_ids, lambda a, b: similarity[sim_col[a], sim_col[b]]
)
novelty_item_item = novelty(item_item_ids, popularity.counts)
novelty_popularity = novelty(popularity_ids, popularity.counts)   # LOWER (common books)

beyond_accuracy_explanation = (
    "MF ranks best on hit@k (Question 3) yet reaches only ~7% of the catalog, while item-item CF "
    "spreads across ~49% -- about 6.6x wider reach. Two paths with similar hit@k can still serve "
    "very different readers because hit@k is silent about WHICH books fill the list: one path can "
    "keep recommending the same narrow head to everyone while another reaches deep into the tail. "
    "Popularity shows the extreme -- it serves the best-known books to all readers, so its picks "
    "carry the LEAST novelty (lowest mean self-information): crowd-pleasers, not discoveries."
)

print(f"catalog coverage  mf: {mf_coverage:.4f}   item-item: {item_item_coverage:.4f}")
print(f"warm reader {reader}  diversity(item-item top-10): {diversity_item_item:.4f}")
print(f"novelty  item-item: {novelty_item_item:.4f}   popularity: {novelty_popularity:.4f}")

# item-item reaches far more of the catalog than MF, despite MF's higher hit@k.
assert item_item_coverage > 5 * mf_coverage, "item-item must cover far more catalog than MF"
assert 0.0 < diversity_item_item <= 1.0, "intra-list diversity is a 1 - mean-similarity in (0, 1]"
assert novelty_popularity < novelty_item_item, "popularity's crowd-pleasers are the least novel"

## Question 5 -- Build the blend on `val`, then FREEZE it

Now assemble the first blended recommender. `blend` calibrates each path's scores to a common [0, 1]
scale, takes the weighted union of their candidate pools, and `rank(exclude=seen)` keeps the top k.
Two choices shape it: the per-path **retrieval pool** (how many candidates each path contributes) and
the per-path **weights**. Measure the result on `val` with `run_blended_scoreboard`: report the
blend's hit@10 and catalog coverage, and compare against matrix factorization, the best single path
on hit@10.

The pool and weights below are the configuration measured on this data in Unit 6 (pool 30; popularity
down-weighted because, as Question 6 shows, it adds neither accuracy nor coverage -- it is kept only
for cold-reader robustness). Measure the blend on `val`, then **FREEZE** every decision into
`FROZEN_PATHS`, `FROZEN_WEIGHTS`, and `FROZEN_POOL`, and record the frozen `val` hit@10 -- Question 7
scores exactly this recommender on `test` and changes nothing. In two or three sentences, say
honestly whether the blend beats the best single path here, and why "a blend always wins" would be
the wrong lesson to draw.

In [ ]:
FROZEN_POOL = 30
FROZEN_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
FROZEN_PATHS = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}

blend_val = run_blended_scoreboard(
    FROZEN_PATHS, interactions_path, catalog_ids=catalog_ids,
    k=K, pool=FROZEN_POOL, weights=FROZEN_WEIGHTS, cold_readers=cold_readers,
)
blend_val_hit = blend_val.hit_rate_at_k        # ~0.306
blend_val_coverage = blend_val.catalog_coverage  # ~0.333

mf_val = run_blended_scoreboard(
    {"mf": mf}, interactions_path, catalog_ids=catalog_ids,
    k=K, pool=FROZEN_POOL, weights={"mf": 1.0}, cold_readers=cold_readers,
)
mf_val_hit = mf_val.hit_rate_at_k              # ~0.276
blend_beats_best_single = bool(blend_val_hit >= mf_val_hit)

blend_reading = (
    "On this data at this PINNED config the blend wins on both axes: hit@10 ~0.306 edges past MF's "
    "~0.276 (the best single path), and catalog coverage ~0.333 is about 4.5x MF's ~0.0745 -- so "
    "it ranks a little better AND reaches far more of the shelf. 'A blend always wins' would be "
    "the wrong lesson: this is a result HERE, not a law; a different dataset, pool, or weight "
    "vector could hand the crown back to a single path. `val` is the split we select on, but the "
    "pool and weights were adopted and frozen -- not searched to maximize val, and certainly not "
    "tuned against the test score Question 7 reads -- so the number is an honest estimate, not a "
    "val-sweep maximum reported as expected performance."
)

print(f"blend  hit@10 {blend_val_hit:.3f}   coverage {blend_val_coverage:.4f}")
print(f"MF     hit@10 {mf_val_hit:.3f}   coverage {mf_coverage:.4f}")
print(f"coverage ratio blend / MF: {blend_val_coverage / mf_coverage:.2f}x")
print(f"blend beats best single path on hit@10: {blend_beats_best_single}")

# The pinned-config blend clears the best single path on hit@10 and covers >= 3x its catalog.
assert blend_val_hit >= mf_val_hit, "pinned blend hit@10 is at least MF's (~0.306 vs 0.276)"
assert abs(blend_val_hit - 0.306) < 0.02, "the frozen val hit@10 is about 0.306"
assert blend_val_coverage >= 3 * mf_coverage, "the blend covers >= 3x MF's catalog (~0.333 vs 0.0745)"
assert blend_beats_best_single

## Question 6 -- Does every path earn its place? The leave-one-path-out ablation

A blend can carry dead weight. Measure each path's marginal value with a **leave-one-path-out
ablation** on `val`: drop one path, re-blend the other three at the same pool and their same weights,
and record the signed change in hit@10. A path that genuinely contributes makes hit@10 *fall* when it
is removed.

You will find that dropping lexical, item-item, or MF each lowers hit@10, but dropping **popularity**
does not -- its marginal accuracy value is about zero. In three or four sentences, interpret the
ablation and explain the mechanism: min-max **calibration** maps every path's top candidate to a
score of 1.0, so a reader-independent popularity path ties its head books with every reader's true
top pick, which is why its *measured* marginal hit@10 is about zero and "every path contributes" is
false here. (Given context, not a graded requirement: the blend still keeps popularity at a small
weight for cold-reader robustness -- it is the one path that serves readers whose empty history
leaves the personalized paths returning `[]`.)

In [ ]:
ablation_rows = []
for dropped in FROZEN_PATHS:
    kept = {name: path for name, path in FROZEN_PATHS.items() if name != dropped}
    kept_weights = {name: FROZEN_WEIGHTS[name] for name in kept}
    board = run_blended_scoreboard(
        kept, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=FROZEN_POOL, weights=kept_weights, cold_readers=cold_readers,
    )
    ablation_rows.append({
        "dropped": dropped,
        "blend_hit@10": board.hit_rate_at_k,
        "delta_vs_full": board.hit_rate_at_k - blend_val_hit,   # signed
    })

delta_by_drop = {row["dropped"]: row["delta_vs_full"] for row in ablation_rows}
dropping_popularity_does_not_hurt = bool(delta_by_drop["popularity"] >= 0.0)

ablation_explanation = (
    "Dropping lexical, item-item, or MF each LOWERS hit@10 -- those paths earn their place -- but "
    "dropping popularity does not: its marginal accuracy value is about zero (hit@10 nudges up, "
    "not down). The mechanism is min-max CALIBRATION: blend rescales each path's pool on its own, "
    "so every path's top candidate becomes score 1.0, and a reader-INDEPENDENT popularity path "
    "therefore ties its handful of head books with every reader's true top pick -- crowding the "
    "top of the pool without being right for anyone in particular. So 'every path contributes' is "
    "false here; what settles a path's value is its MEASURED leave-one-out change, and popularity's "
    "is about zero. The blend keeps popularity at a small 0.25 anyway for one reason -- cold-reader "
    "robustness, not accuracy or coverage: it still serves the readers whose empty history leaves "
    "the personalized paths returning an empty list."
)

print(f"full blend hit@10 = {blend_val_hit:.3f}")
for row in ablation_rows:
    arrow = "UP  " if row["delta_vs_full"] > 0 else "down"
    print(f"  drop {row['dropped']:10s} -> {row['blend_hit@10']:.3f}  "
          f"(delta {row['delta_vs_full']:+.3f}  {arrow})")
print(f"dropping popularity does not hurt hit@10: {dropping_popularity_does_not_hurt}")

# The surprise: dropping the reader-independent popularity path does NOT lower hit@10 ...
assert dropping_popularity_does_not_hurt, "dropping popularity must not lower hit@10 here"
# ... while dropping a reader-dependent path (MF) clearly does.
assert delta_by_drop["mf"] < 0.0, "dropping the reader-dependent MF path must lower hit@10"
assert delta_by_drop["item-item"] < 0.0, "dropping item-item must lower hit@10 too"

## Question 7 -- Unseal `test`: one honest final score

The `test` split has been sealed since Unit 1. Now -- and only now, with every decision frozen in
Question 5 -- score the FROZEN recommender on `test` exactly once. Pass `split="test"` to
`run_blended_scoreboard` with the frozen paths, weights, and pool, and the same cold-reader exclusion.
Report the test hit@10, the test catalog coverage, and the number of readers scored, and compare them
to the `val` numbers you froze in Question 5.

Then write three or four sentences of honest interpretation. How do the `val` and `test` scores
compare, and roughly how many held-out readers does each aggregate over? And the hygiene question at
the heart of this checkpoint: why do you evaluate `test` exactly once and never tune any decision
against it -- what would the final number stop meaning if you went back and adjusted the blend after
seeing it?

In [ ]:
# The authorized one-shot unseal of the sealed `test` holdout. Run this ONCE; do not return to
# Questions 1-6 to change any decision after reading it.
blend_test = run_blended_scoreboard(
    FROZEN_PATHS, interactions_path, catalog_ids=catalog_ids,
    k=K, pool=FROZEN_POOL, weights=FROZEN_WEIGHTS, cold_readers=cold_readers, split="test",
)
test_hit = blend_test.hit_rate_at_k
test_coverage = blend_test.catalog_coverage
test_readers = blend_test.readers

val_vs_test_hit = (blend_val_hit, test_hit)

test_interpretation = (
    "The frozen recommender scores hit@10 ~0.306 on test, essentially the same as its ~0.306 on "
    "val, with comparable coverage (~0.34) -- an honest sign the val estimate was not an "
    "optimistic fluke. Each number aggregates over several hundred held-out readers -- here 500 on "
    "test after the 60 cold readers (and readers with no unseen positive) are excluded, a similar "
    "count on val -- so both are stable averages, not noise. We evaluate test exactly ONCE and "
    "never tune a decision against "
    "it because the moment we adjusted the blend after seeing the test number, test would become a "
    "second validation set: the final figure would measure how well we fit THIS holdout, not how "
    "the recommender performs on genuinely unseen readers -- which is the only thing it is for."
)

print(f"val  hit@10 {blend_val_hit:.3f}   coverage {blend_val_coverage:.4f}")
print(f"test hit@10 {test_hit:.3f}   coverage {test_coverage:.4f}   readers {test_readers}")
print(f"val vs test hit@10: {val_vs_test_hit}")

# The one-time test evaluation runs and returns a finite hit@10 close to the frozen val number.
assert np.isfinite(test_hit), "the test evaluation must return a finite hit@10"
assert abs(test_hit - blend_val_hit) < 0.05, "val and test hit@10 should be close (honest estimate)"
assert test_readers > 300, "the test score aggregates over several hundred held-out readers"